# 01 — Data Exploration
### Multimodal Disease Detection Framework
**Paper**: *Optimizing Multimodal Deep Learning Architectures for Early Disease Detection*

This notebook explores the three datasets used in the paper:
- **ChestX-ray14** — Medical imaging
- **MIMIC-III** — Structured EHR
- **PhysioNet ECG** — Sequential time-series

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import torch
import sys
sys.path.insert(0, '../src')

plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('deep')
print('Libraries loaded.')

## 1. Dataset Statistics (Table 1 from Paper)

In [ ]:
# Dataset composition — Table 1 from paper
datasets = pd.DataFrame({
    'Dataset':            ['ChestX-ray14', 'MIMIC-III', 'PhysioNet ECG'],
    'Modality':           ['Imaging (X-ray)', 'Structured EHR', 'Time-series ECG'],
    'Total Samples':      [112120, 58976, 10000],
    'Patients':           [30000, 38500, 6000],
    'Positive Class (%)': [22.1, 19.3, 26.7],
    'Split':              ['70/20/10', '70/20/10', '70/20/10']
})
display(datasets)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# Sample counts
axes[0].bar(datasets['Dataset'], datasets['Total Samples'],
            color=['#2196F3', '#4CAF50', '#F44336'], edgecolor='black')
axes[0].set_title('Total Samples per Dataset', fontweight='bold')
axes[0].set_ylabel('Number of Samples')
axes[0].tick_params(axis='x', rotation=15)

# Class imbalance
for i, (_, row) in enumerate(datasets.iterrows()):
    pos = row['Positive Class (%)']
    axes[1].bar(row['Dataset'], pos, color=['#2196F3', '#4CAF50', '#F44336'][i], edgecolor='black')
axes[1].axhline(y=25, color='orange', linestyle='--', label='25% threshold')
axes[1].set_title('Positive Class Prevalence (%)', fontweight='bold')
axes[1].set_ylabel('Positive Class (%)')
axes[1].tick_params(axis='x', rotation=15)
axes[1].legend()

# Patient counts
axes[2].bar(datasets['Dataset'], datasets['Patients'],
            color=['#2196F3', '#4CAF50', '#F44336'], edgecolor='black')
axes[2].set_title('Unique Patients per Dataset', fontweight='bold')
axes[2].set_ylabel('Number of Patients')
axes[2].tick_params(axis='x', rotation=15)

plt.suptitle('Dataset Characteristics (Table 1 from Paper)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('../results/dataset_characteristics.png', dpi=150, bbox_inches='tight')
plt.show()

## 2. Simulated ECG Signal Visualization

In [ ]:
# Simulate ECG-like signal to visualize preprocessing
import sys
sys.path.insert(0, '../src')
from data.preprocess import ECGPreprocessor

np.random.seed(42)
t = np.linspace(0, 10, 5000)
# Synthetic ECG: QRS complex approximation
raw_ecg = np.sin(2 * np.pi * 1.2 * t) + 0.3 * np.sin(2 * np.pi * 8 * t) + 0.5 * np.random.randn(5000)
raw_ecg = raw_ecg[:, np.newaxis]  # (5000, 1)

proc = ECGPreprocessor(sampling_rate=500, window_seconds=5)
clean_ecg = proc.preprocess(raw_ecg, random_crop=False)

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 6))
ax1.plot(t, raw_ecg[:, 0], color='#F44336', alpha=0.8, linewidth=0.8)
ax1.set_title('Raw ECG Signal (with noise artifacts)', fontweight='bold')
ax1.set_xlabel('Time (s)')
ax1.set_ylabel('Amplitude (mV)')

t2 = np.linspace(0, 5, len(clean_ecg))
ax2.plot(t2, clean_ecg[:, 0], color='#2196F3', alpha=0.9, linewidth=0.8)
ax2.set_title('Preprocessed ECG: Filtered + Normalized + Windowed (60s)', fontweight='bold')
ax2.set_xlabel('Time (s)')
ax2.set_ylabel('Normalized Amplitude')

plt.tight_layout()
plt.savefig('../results/ecg_preprocessing.png', dpi=150, bbox_inches='tight')
plt.show()

## 3. EHR Feature Distribution

In [ ]:
# Simulate EHR features inspired by MIMIC-III clinical variables
np.random.seed(0)
N = 500
ehr_df = pd.DataFrame({
    'Age':          np.random.normal(62, 15, N).clip(18, 95),
    'Troponin':     np.random.exponential(0.5, N).clip(0, 5),
    'CRP':          np.random.exponential(8, N).clip(0, 80),
    'Heart Rate':   np.random.normal(78, 18, N).clip(40, 180),
    'SpO2':         np.random.normal(96, 3, N).clip(80, 100),
    'Creatinine':   np.random.normal(1.1, 0.5, N).clip(0.3, 8),
})

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
for ax, col in zip(axes.flat, ehr_df.columns):
    ax.hist(ehr_df[col], bins=30, color='#4CAF50', edgecolor='white', alpha=0.8)
    ax.set_title(f'{col} Distribution', fontweight='bold')
    ax.set_xlabel(col)
    ax.set_ylabel('Count')

plt.suptitle('EHR Feature Distributions (MIMIC-III Inspired)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('../results/ehr_distributions.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. SHAP Feature Importance (Table 9 from Paper)

In [ ]:
# Reproduce Table 9 from the paper — SHAP contributions
shap_data = pd.DataFrame({
    'Feature / Modality':    ['Troponin levels (EHR)', 'Imaging lesion intensity',
                              'HR variability (ECG)', 'Age', 'CRP level (EHR)'],
    'SHAP Contribution':     [0.48, 0.44, 0.39, 0.31, 0.27],
    'Modality':              ['EHR', 'Imaging', 'ECG', 'EHR', 'EHR'],
    'Clinical Insight':      [
        'Indicative of cardiac event risk',
        'Associated with pneumonia severity',
        'Linked to cardiovascular deterioration',
        'Older patients at higher risk',
        'Correlates with infection progression'
    ]
})

colors = {'EHR': '#4CAF50', 'Imaging': '#2196F3', 'ECG': '#F44336'}
bar_colors = [colors[m] for m in shap_data['Modality']]

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.barh(shap_data['Feature / Modality'], shap_data['SHAP Contribution'],
               color=bar_colors, edgecolor='black')
ax.set_xlabel('Mean |SHAP Value|', fontsize=12)
ax.set_title('Top Contributing Features (Table 9 from Paper)', fontsize=13, fontweight='bold')
ax.invert_yaxis()

for bar, val in zip(bars, shap_data['SHAP Contribution']):
    ax.text(val + 0.005, bar.get_y() + bar.get_height()/2,
            f'{val:.2f}', va='center', fontweight='bold')

# Legend
from matplotlib.patches import Patch
legend_elements = [Patch(facecolor=v, label=k) for k, v in colors.items()]
ax.legend(handles=legend_elements, loc='lower right')

plt.tight_layout()
plt.savefig('../results/shap_features.png', dpi=150, bbox_inches='tight')
plt.show()
display(shap_data[['Feature / Modality', 'SHAP Contribution', 'Modality', 'Clinical Insight']])